# Phase 3: LoRA fine-tuning of Qwen2-VL-2B on native-Windows GUI grounding (v1)

First real training run for the `computer-use` project's GUI grounding model (ADR-0003). Runs on Kaggle's free T4/P100 GPU quota, matching the project's $0-budget commitment.

**Honest scope, stated up front**: this trains on the **v1 dataset** -- 528 examples across 8/14 registry apps, frozen 2026-07-13 (`data/gui_grounding/README.md`). The held-out pool is currently 1 app (Character Map, rich-tree only), so **do not treat this run's held-out numbers as a completed H1-H3 test** -- they're provisional until the remaining apps are collected. The point of this run is to prove the training loop actually works end-to-end and get a first real dev-loss number, not to report a final result.

Every piece below (`prepare_dataset.py`, `chat_format.py`, `lora_config.py`, `dataset.py`, `train_lora.py`) was built and verified locally (97 passing tests, plus live checks against the real tokenizer/processor/model architecture on the `meta` device) before this notebook was written -- see `docs/journal.md` for that verification trail. This notebook is the first time any of it touches real model weights or a GPU.

## 1. Confirm GPU is attached

In Kaggle: Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100). Must be set before running anything below.

In [ ]:
!nvidia-smi

## 2. Get the code

Clones the public `computer-use` repo and installs it with the `training` extra (`transformers`, `torch`, `torchvision`, `peft`, `jinja2` -- see `pyproject.toml`).

In [ ]:
!git clone https://github.com/rudranaresh0201/computer-use.git
%cd computer-use
!pip install -q -e ".[training]"

## 3. Attach the v1 dataset

Upload `data/gui_grounding/` (the `images/`, `labels.jsonl`, and `splits/` folders -- NOT committed to git, since they're generated data, not source) as a Kaggle Dataset before running this notebook, then attach it via the right panel ("+ Add Input"). It will be mounted read-only at `/kaggle/input/<your-dataset-slug>/`.

Set `KAGGLE_DATASET_ROOT` below to match whatever slug you gave it.

In [ ]:
from pathlib import Path

KAGGLE_DATASET_ROOT = Path("/kaggle/input/gui-grounding-v1")  # <-- adjust to your uploaded dataset's slug
OUTPUT_DIR = Path("/kaggle/working/lora_grounder")

assert (KAGGLE_DATASET_ROOT / "labels.jsonl").exists(), (
    f"labels.jsonl not found under {KAGGLE_DATASET_ROOT} -- check the dataset is attached "
    "and KAGGLE_DATASET_ROOT matches its slug"
)
assert (KAGGLE_DATASET_ROOT / "splits" / "train.jsonl").exists(), (
    "splits/train.jsonl not found -- run training/prepare_dataset.py locally first "
    "and include its splits/ output when you upload the Kaggle Dataset"
)
print("dataset root looks correct:", KAGGLE_DATASET_ROOT)

## 4. Build the trainer

Loads the real Qwen2-VL-2B-Instruct weights (~4GB download, first cell run only), attaches the verified LoRA config (0.829% trainable params against the real architecture -- see `training/lora_config.py`), and wires up `GroundingDataset`/`collate_fn` against the v1 train/dev splits.

In [ ]:
from computeruse.training.train_lora import build_trainer

trainer = build_trainer(KAGGLE_DATASET_ROOT, OUTPUT_DIR)

## 5. Train

1 epoch over 266 train examples (v1's actual size) is fast even on a single T4 -- this is deliberately a small first run, not the final tuned config. Watch the eval (dev) loss printed each epoch; per the hypothesis doc, **only the dev split is allowed to drive any tuning** -- do not look at `test_held_out_app` or `test_same_app` numbers to make decisions here.

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
trainer.save_model(str(OUTPUT_DIR / "final"))
print("saved LoRA adapter to", OUTPUT_DIR / "final")

## 6. Sanity-check a few predictions

Not a real evaluation (that's the 4-arm ablation, not built yet) -- just a first look at whether the model outputs something plausible at all: a well-formed `(x,y)` string in the right numeric range, before any accuracy metric is computed.

In [ ]:
import json

from PIL import Image
from transformers import AutoProcessor

from computeruse.training.chat_format import to_conversation
from computeruse.training.prepare_dataset import TrainingExample

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
model = trainer.model
model.eval()

dev_examples = [
    TrainingExample(**json.loads(line))
    for line in (KAGGLE_DATASET_ROOT / "splits" / "dev.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
][:5]

print(f"spot-checking {len(dev_examples)} dev examples (not a metric, just a look)")
for ex in dev_examples:
    image = Image.open(KAGGLE_DATASET_ROOT / ex.image_path).convert("RGB")
    prompt_only = to_conversation(ex)[:-1]  # drop the ground-truth assistant turn
    prompt_text = processor.tokenizer.apply_chat_template(
        prompt_only, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to(model.device)
    generated = model.generate(**inputs, max_new_tokens=16)
    prediction = processor.tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"{ex.app:12s} {ex.prompt!r:35s} predicted={prediction!r}  ground_truth={ex.target!r}")

## Next steps (not this notebook)

1. Collect the 6 blocked apps (needs an admin/elevated session + 2 UAC clicks) and re-run `prepare_dataset.py` before any real held-out evaluation.
2. Build the 4 evaluation arms (UIA-only, zero-shot VLM, fine-tuned grounder, hybrid) -- none of that exists yet.
3. Only then run the one final held-out evaluation and report H1-H3, sliced 3 ways, honestly.